# Sinhala Buddhist Corpus - Complete Analysis & Preprocessing v4

This notebook performs:
1. **Pre-processing Analysis** - Statistics on raw extracted text
2. **Data Cleaning** - Remove English text and URLs  
3. **Language Separation** - Separate Sinhala from Pali using pattern-based classification
4. **Post-processing Analysis** - Compare before/after statistics
5. **Corpus Generation** - Create final training-ready text files

**New in v4:**
- ✅ Robust pattern-based Sinhala/Pali classifier
- ✅ Extensive grammatical pattern matching
- ✅ Confidence scoring for classifications
- ✅ Detailed per-PDF language distribution analysis

## 1. Setup and Configuration

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import re
import json
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported")

✓ Libraries imported


In [3]:
# Directory configuration
BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/")

# Input directories (from Document AI extraction)
EXTRACTION_BASE = BASE_DIR / "data" / "docai_extractions"
RAW_TEXT_DIR = EXTRACTION_BASE / "1_raw_text"
CLEANED_TEXT_DIR = EXTRACTION_BASE / "2_cleaned_text"

# Output directories (preprocessed corpus)
PREPROCESSED_DIR = BASE_DIR / "data" / "preprocessed"
ANALYSIS_DIR = PREPROCESSED_DIR / "analysis"
SINHALA_CORPUS_FILE = PREPROCESSED_DIR / "sinhala_text_corpus.txt"
PALI_CORPUS_FILE = PREPROCESSED_DIR / "pali_text_corpus.txt"
MIXED_CORPUS_FILE = PREPROCESSED_DIR / "mixed_text_corpus.txt"

# Create directories
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Directory structure created")
print(f"  Input (cleaned text): {CLEANED_TEXT_DIR}")
print(f"  Output (preprocessed): {PREPROCESSED_DIR}")
print(f"  Analysis results: {ANALYSIS_DIR}")

✓ Directory structure created
  Input (cleaned text): /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/docai_extractions/2_cleaned_text
  Output (preprocessed): /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/preprocessed
  Analysis results: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/preprocessed/analysis


## 2. Sinhala/Pali Pattern-Based Classifier

This classifier uses grammatical patterns to distinguish between:
- **Sinhala** - Modern Sinhala with particles like වූ, හි, ට, ගේ
- **Pali** - Pali transliterations with endings like -තෝ, -ස්ස, -ති, -න්ති

In [4]:
class SinhalaPaliClassifier:
    """
    Classify text as Sinhala or Pali transliteration using grammatical patterns.
    """

    def __init__(self):
        # Pali grammatical endings
        self.pali_endings = [
            r'තෝ\b',      # Genitive: භගවතෝ
            r'ස්ස\b',     # Genitive: තථාගතස්ස
            r'නො\b',      # Nominative plural: සබ්බඤ්ඤුනො
            r'ාය\b',      # Dative: බුද්ධාය
            r'ානං\b',     # Genitive plural
            r'ස්මා\b',    # Ablative
            r'ස්මිං\b',   # Locative
            r'ති\b',      # Verb 3rd person
            r'න්ති\b',    # Verb 3rd plural
            r'තේ\b',      # Verb middle
            r'න්තේ\b',    # Verb middle plural
            r'ම්හි\b',    # 1st person plural
        ]

        # Sinhala grammatical particles and case markers
        self.sinhala_particles = [
            r'\bවූ\b',     # Past participle
            r'\bවන\b',     # Present participle
            r'\bකළ\b',     # Past
            r'\bගත්\b',    # Past
            r'\bදුන්\b',   # Past
            r'හි\b',       # Locative
            r'ට\b',        # Dative
            r'ගේ\b',       # Genitive
            r'යේ\b',       # Locative
            r'ෙයන්\b',    # Instrumental
            r'යි\b',       # Emphatic
            r'\bද\b',      # Question
            r'නේ\b',       # Question
            r'වා\b',       # Conjunctive
            r'මු\b',       # 1st person past
            r'මි\b',       # 1st person present
            r'හ\b',        # Plural
        ]

        # Known Pali words
        self.pali_words = [
            'භගවතෝ', 'තථාගතස්ස', 'සබ්බඤ්ඤුනො', 'සම්මාසම්බුද්ධස්ස',
            'ධම්මස්ස', 'සඞ්ඝස්ස', 'භික්ඛුනො', 'භික්ඛූනං',
            'සුඛස්ස', 'දුක්ඛස්ස', 'නමො', 'භන්තේ',
        ]

        # Known Sinhala words
        self.sinhala_words = [
            'මාගේ', 'අපගේ', 'ඔබගේ', 'එසේ', 'මෙසේ',
            'කියන', 'කියා', 'කරන', 'කරමු', 'බව',
            'නිසා', 'හෙයින්', 'සිට', 'දක්වා', 'පටන්', 'තෙක්',
            'පිළිබඳ', 'අනුව', 'විසින්', 'කෙරෙහි', 'තුළ',
        ]

        # Compile patterns
        self.pali_ending_pattern = re.compile('|'.join(self.pali_endings))
        self.sinhala_particle_pattern = re.compile('|'.join(self.sinhala_particles))

    def count_pali_features(self, text: str) -> int:
        count = len(self.pali_ending_pattern.findall(text))
        for word in self.pali_words:
            count += text.count(word)
        return count

    def count_sinhala_features(self, text: str) -> int:
        count = len(self.sinhala_particle_pattern.findall(text))
        for word in self.sinhala_words:
            count += text.count(word)
        return count

    def classify(self, text: str, threshold: float = 0.6) -> Tuple[str, Dict]:
        """
        Classify text as 'sinhala', 'pali', or 'mixed'.

        Returns:
            Tuple of (language, debug_info)
        """
        if not text or len(text.strip()) < 50:
            return 'mixed', {'reason': 'Text too short', 'confidence': 0.0}

        pali_features = self.count_pali_features(text)
        sinhala_features = self.count_sinhala_features(text)
        total_features = pali_features + sinhala_features

        debug_info = {
            'text_length': len(text),
            'pali_features': pali_features,
            'sinhala_features': sinhala_features,
            'total_features': total_features
        }

        # No features detected
        if total_features == 0:
            # Check for very long compound words (Pali characteristic)
            words = text.split()
            long_words = [w for w in words if len(w) > 25]

            if len(long_words) > 2:
                debug_info['reason'] = 'Long compound words (Pali)'
                debug_info['confidence'] = 0.6
                return 'pali', debug_info

            debug_info['reason'] = 'No distinctive features'
            debug_info['confidence'] = 0.0
            return 'mixed', debug_info

        # Calculate ratios
        pali_ratio = pali_features / total_features
        sinhala_ratio = sinhala_features / total_features

        debug_info['pali_ratio'] = pali_ratio
        debug_info['sinhala_ratio'] = sinhala_ratio

        # Decision logic
        if sinhala_ratio >= threshold:
            debug_info['confidence'] = sinhala_ratio
            debug_info['reason'] = f'Strong Sinhala ({sinhala_ratio:.1%})'
            return 'sinhala', debug_info
        elif pali_ratio >= threshold:
            debug_info['confidence'] = pali_ratio
            debug_info['reason'] = f'Strong Pali ({pali_ratio:.1%})'
            return 'pali', debug_info
        else:
            # Use majority
            if sinhala_features > pali_features:
                debug_info['confidence'] = sinhala_ratio
                debug_info['reason'] = f'Moderate Sinhala ({sinhala_ratio:.1%})'
                return 'sinhala', debug_info
            else:
                debug_info['confidence'] = pali_ratio
                debug_info['reason'] = f'Moderate Pali ({pali_ratio:.1%})'
                return 'pali', debug_info


# Initialize classifier
language_classifier = SinhalaPaliClassifier()
print("✓ Sinhala/Pali classifier initialized")

✓ Sinhala/Pali classifier initialized


## 3. Data Loading Functions

In [5]:
def load_all_text_files(text_dir: Path) -> Dict[str, List[str]]:
    """
    Load all text files organized by PDF.

    Returns:
        Dict mapping pdf_name -> list of page texts
    """
    corpus = {}

    pdf_dirs = [d for d in text_dir.iterdir() if d.is_dir()]

    for pdf_dir in tqdm(pdf_dirs, desc="Loading text files"):
        pdf_name = pdf_dir.name
        page_texts = []

        txt_files = [f for f in pdf_dir.glob("*.txt")]

        def get_page_number(file_path):
            try:
                stem = file_path.stem
                if '_' in stem:
                    return int(stem.split('_')[-1])
                else:
                    return int(stem)
            except:
                return 0

        txt_files_sorted = sorted(txt_files, key=get_page_number)

        for page_file in txt_files_sorted:
            try:
                with open(page_file, 'r', encoding='utf-8') as f:
                    text = f.read().strip()
                    if text:  # Only include non-empty pages
                        page_texts.append(text)
            except Exception as e:
                print(f"  Warning: Error reading {page_file}: {e}")

        if page_texts:
            corpus[pdf_name] = page_texts

    return corpus


print("✓ Data loading functions defined")

✓ Data loading functions defined


## 4. Analysis Functions

In [6]:
def analyze_corpus(corpus: Dict[str, List[str]], corpus_name: str = "Corpus") -> Dict:
    """
    Comprehensive statistical analysis of corpus.
    """
    print(f"\n{'='*70}")
    print(f"{corpus_name.upper()} ANALYSIS")
    print(f"{'='*70}\n")

    stats = {
        'corpus_name': corpus_name,
        'num_pdfs': len(corpus),
        'num_pages': 0,
        'total_characters': 0,
        'total_words': 0,
        'page_word_counts': [],
        'page_char_counts': [],
        'pdf_stats': []
    }

    for pdf_name, page_texts in corpus.items():
        pdf_word_count = 0
        pdf_char_count = 0

        for page_text in page_texts:
            char_count = len(page_text)
            word_count = len(page_text.split())

            pdf_char_count += char_count
            pdf_word_count += word_count

            stats['page_char_counts'].append(char_count)
            stats['page_word_counts'].append(word_count)

        stats['pdf_stats'].append({
            'pdf_name': pdf_name,
            'pages': len(page_texts),
            'words': pdf_word_count,
            'characters': pdf_char_count
        })

        stats['num_pages'] += len(page_texts)
        stats['total_words'] += pdf_word_count
        stats['total_characters'] += pdf_char_count

    # Calculate statistics
    word_counts = np.array(stats['page_word_counts'])

    stats['words_per_page'] = {
        'mean': float(np.mean(word_counts)),
        'median': float(np.median(word_counts)),
        'std': float(np.std(word_counts)),
        'min': int(np.min(word_counts)),
        'max': int(np.max(word_counts)),
    }

    # Print summary
    print(f"📚 Corpus Overview:")
    print(f"   PDFs: {stats['num_pdfs']}")
    print(f"   Total Pages: {stats['num_pages']:,}")
    print(f"   Total Words: {stats['total_words']:,}")
    print(f"   Total Characters: {stats['total_characters']:,}")

    print(f"\n📊 Words per Page:")
    print(f"   Mean: {stats['words_per_page']['mean']:.1f}")
    print(f"   Median: {stats['words_per_page']['median']:.1f}")
    print(f"   Std Dev: {stats['words_per_page']['std']:.1f}")
    print(f"   Min: {stats['words_per_page']['min']}")
    print(f"   Max: {stats['words_per_page']['max']}")

    return stats


print("✓ Analysis functions defined")

✓ Analysis functions defined


## 5. Text Cleaning Functions

In [7]:
def is_latin_char(char: str) -> bool:
    """Check if character is Latin (English)."""
    code = ord(char)
    return (0x0041 <= code <= 0x005A) or (0x0061 <= code <= 0x007A)


def remove_urls(text: str) -> str:
    """Remove URLs from text."""
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    return text


def remove_english_words(text: str) -> str:
    """Remove English words from text."""
    words = text.split()
    filtered_words = []

    for word in words:
        clean_word = word.strip('.,;:!?\'"()[]{}-')

        if not clean_word:
            continue

        # Check if word is purely Latin
        latin_chars = sum(1 for c in clean_word if is_latin_char(c))
        total_alpha = sum(1 for c in clean_word if c.isalpha())

        # If >80% Latin, skip
        if total_alpha > 0 and (latin_chars / total_alpha) > 0.8:
            continue

        filtered_words.append(word)

    return ' '.join(filtered_words)


def clean_text(text: str) -> str:
    """Complete text cleaning pipeline."""
    text = remove_urls(text)
    text = remove_english_words(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    return text.strip()


print("✓ Text cleaning functions defined")

✓ Text cleaning functions defined


## 6. Language Separation Functions

In [8]:
def separate_languages(
    corpus: Dict[str, List[str]],
    classifier: SinhalaPaliClassifier,
    threshold: float = 0.6
) -> Tuple[List[str], List[str], List[str], Dict]:
    """
    Separate corpus into Sinhala, Pali, and Mixed texts.

    Returns:
        Tuple of (sinhala_texts, pali_texts, mixed_texts, statistics)
    """
    sinhala_texts = []
    pali_texts = []
    mixed_texts = []

    stats = {
        'total_pages': 0,
        'sinhala_pages': 0,
        'pali_pages': 0,
        'mixed_pages': 0,
        'pdf_classifications': defaultdict(lambda: {'sinhala': 0, 'pali': 0, 'mixed': 0}),
        'classification_reasons': defaultdict(int),
        'sample_classifications': []  # Store first 50 for review
    }

    print("\nClassifying pages by language...")

    for pdf_name, page_texts in tqdm(corpus.items(), desc="Processing PDFs"):
        for page_num, page_text in enumerate(page_texts, 1):
            stats['total_pages'] += 1

            # Classify this page
            language, debug_info = classifier.classify(page_text, threshold=threshold)

            # Track statistics
            reason = debug_info.get('reason', 'unknown')
            stats['classification_reasons'][reason] += 1

            # Store sample classifications
            if len(stats['sample_classifications']) < 50:
                stats['sample_classifications'].append({
                    'pdf': pdf_name,
                    'page': page_num,
                    'language': language,
                    'confidence': debug_info.get('confidence', 0.0),
                    'reason': reason,
                    'text_preview': page_text[:150]
                })

            # Categorize
            if language == 'sinhala':
                sinhala_texts.append(page_text)
                stats['sinhala_pages'] += 1
                stats['pdf_classifications'][pdf_name]['sinhala'] += 1
            elif language == 'pali':
                pali_texts.append(page_text)
                stats['pali_pages'] += 1
                stats['pdf_classifications'][pdf_name]['pali'] += 1
            else:  # mixed
                mixed_texts.append(page_text)
                stats['mixed_pages'] += 1
                stats['pdf_classifications'][pdf_name]['mixed'] += 1

    print(f"\n✓ Language separation complete")
    print(f"   Total pages: {stats['total_pages']}")
    print(f"   Sinhala pages: {stats['sinhala_pages']} ({stats['sinhala_pages']/stats['total_pages']*100:.1f}%)")
    print(f"   Pali pages: {stats['pali_pages']} ({stats['pali_pages']/stats['total_pages']*100:.1f}%)")
    print(f"   Mixed pages: {stats['mixed_pages']} ({stats['mixed_pages']/stats['total_pages']*100:.1f}%)")

    return sinhala_texts, pali_texts, mixed_texts, dict(stats)


print("✓ Language separation functions defined")

✓ Language separation functions defined


## 7. Load Raw Corpus

In [9]:
print("Loading raw corpus...")
raw_corpus = load_all_text_files(CLEANED_TEXT_DIR)
print(f"✓ Loaded {len(raw_corpus)} PDFs")

Loading raw corpus...


Loading text files:   0%|          | 0/34 [00:00<?, ?it/s]

✓ Loaded 34 PDFs


## 8. Pre-processing Analysis

In [10]:
# Analyze raw corpus
raw_stats = analyze_corpus(raw_corpus, "Raw Corpus (Before Preprocessing)")

# Save statistics
with open(ANALYSIS_DIR / "raw_corpus_stats.json", 'w', encoding='utf-8') as f:
    json.dump(raw_stats, f, indent=2, ensure_ascii=False)

print(f"\n✓ Statistics saved to: {ANALYSIS_DIR / 'raw_corpus_stats.json'}")


RAW CORPUS (BEFORE PREPROCESSING) ANALYSIS

📚 Corpus Overview:
   PDFs: 34
   Total Pages: 10,511
   Total Words: 2,753,232
   Total Characters: 17,819,309

📊 Words per Page:
   Mean: 261.9
   Median: 260.0
   Std Dev: 69.3
   Min: 77
   Max: 563

✓ Statistics saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/preprocessed/analysis/raw_corpus_stats.json


## 9. Clean Corpus

In [11]:
def clean_corpus(corpus: Dict[str, List[str]]) -> Dict[str, List[str]]:
    """Apply cleaning to entire corpus."""
    cleaned_corpus = {}

    print("\nCleaning corpus (removing English and URLs)...")

    for pdf_name, page_texts in tqdm(corpus.items(), desc="Cleaning PDFs"):
        cleaned_pages = []

        for page_text in page_texts:
            cleaned_text = clean_text(page_text)

            if len(cleaned_text.strip()) > 50:
                cleaned_pages.append(cleaned_text)

        if cleaned_pages:
            cleaned_corpus[pdf_name] = cleaned_pages

    print(f"✓ Cleaning complete")
    print(f"   PDFs before: {len(corpus)}")
    print(f"   PDFs after: {len(cleaned_corpus)}")

    return cleaned_corpus


cleaned_corpus = clean_corpus(raw_corpus)


Cleaning corpus (removing English and URLs)...


Cleaning PDFs:   0%|          | 0/34 [00:00<?, ?it/s]

✓ Cleaning complete
   PDFs before: 34
   PDFs after: 34


## 10. Test Classifier on Samples

In [12]:
# Test on first PDF
test_pdf = list(cleaned_corpus.keys())[0]
test_pages = cleaned_corpus[test_pdf][:5]

print(f"\nTesting classifier on first 5 pages from: {test_pdf}")
print("="*80)

for i, page_text in enumerate(test_pages, 1):
    lang, debug = language_classifier.classify(page_text)

    print(f"\nPage {i}:")
    print(f"  Classification: {lang.upper()}")
    print(f"  Confidence: {debug.get('confidence', 0):.1%}")
    print(f"  Reason: {debug.get('reason', 'N/A')}")
    print(f"  Pali features: {debug['pali_features']}, Sinhala features: {debug['sinhala_features']}")
    print(f"  Text preview: {page_text[:120]}...")
    print("-"*80)


Testing classifier on first 5 pages from: අංගුත්තර_නිකාය_1

Page 1:
  Classification: SINHALA
  Confidence: 79.1%
  Reason: Strong Sinhala (79.1%)
  Pali features: 24, Sinhala features: 91
  Text preview: නමෝ තස්ස භගවතෝ සබ්බධම්මේසු අප්පටිහතඤාණචාරස්ස දසබලධරස්ස චතුවේසාරජ්ජවිසාරදස්ස සබ්බසත්තුත්තමස්ස ධම්මිස්සරස්ස ධම්මරාජස්ස ධම්...
--------------------------------------------------------------------------------

Page 2:
  Classification: SINHALA
  Confidence: 91.3%
  Reason: Strong Sinhala (91.3%)
  Pali features: 12, Sinhala features: 126
  Text preview: සංඥාපනය අංගුත්තර නිකාය අප බුදුන් බරණැස්නුවර ඍෂීන් මුළු දෙන මුවළැවහි දී ධර්ම චක්‍ර'ය පැවැත් වූ තැන් පටන් කොට සුභද්‍ර පරිව...
--------------------------------------------------------------------------------

Page 3:
  Classification: SINHALA
  Confidence: 86.7%
  Reason: Strong Sinhala (86.7%)
  Pali features: 28, Sinhala features: 182
  Text preview: හේතු උපමාදීන් ප්‍රතිමණ්ඩිත නානාවිධ දේශනානය සහස්‍රයන් විසින් විචිත්‍ර වන හෙයින් 'විචිත්‍ර ප

## 11. Separate Sinhala and Pali

In [13]:
# Adjustable threshold
CLASSIFICATION_THRESHOLD = 0.6  # 0.0-1.0
# Lower = more lenient (easier to classify as Sinhala/Pali)
# Higher = more strict (more will be mixed)

sinhala_texts, pali_texts, mixed_texts, separation_stats = separate_languages(
    cleaned_corpus,
    language_classifier,
    threshold=CLASSIFICATION_THRESHOLD
)


Classifying pages by language...


Processing PDFs:   0%|          | 0/34 [00:00<?, ?it/s]


✓ Language separation complete
   Total pages: 10511
   Sinhala pages: 10259 (97.6%)
   Pali pages: 252 (2.4%)
   Mixed pages: 0 (0.0%)


In [14]:
# Show classification reasons breakdown
print("\n" + "="*80)
print("CLASSIFICATION REASONS BREAKDOWN")
print("="*80)

reasons_df = pd.DataFrame([
    {
        'Reason': reason,
        'Count': count,
        'Percentage': f"{count/separation_stats['total_pages']*100:.1f}%"
    }
    for reason, count in separation_stats['classification_reasons'].items()
]).sort_values('Count', ascending=False)

print(reasons_df.to_string(index=False))


CLASSIFICATION REASONS BREAKDOWN
                  Reason  Count Percentage
  Strong Sinhala (66.7%)    100       1.0%
 Strong Sinhala (100.0%)     86       0.8%
  Strong Sinhala (95.2%)     77       0.7%
  Strong Sinhala (93.3%)     75       0.7%
  Strong Sinhala (96.2%)     74       0.7%
  Strong Sinhala (75.0%)     73       0.7%
  Strong Sinhala (95.5%)     71       0.7%
  Strong Sinhala (95.7%)     71       0.7%
  Strong Sinhala (95.3%)     70       0.7%
  Strong Sinhala (95.8%)     69       0.7%
  Strong Sinhala (93.5%)     67       0.6%
  Strong Sinhala (91.7%)     67       0.6%
  Strong Sinhala (93.2%)     66       0.6%
  Strong Sinhala (93.8%)     66       0.6%
  Strong Sinhala (96.6%)     64       0.6%
  Strong Sinhala (95.6%)     64       0.6%
  Strong Sinhala (94.0%)     64       0.6%
  Strong Sinhala (97.0%)     63       0.6%
  Strong Sinhala (94.1%)     63       0.6%
  Strong Sinhala (94.4%)     63       0.6%
  Strong Sinhala (96.9%)     63       0.6%
  Strong Sinhala (94

In [15]:
# Show sample classifications
print("\n" + "="*80)
print("SAMPLE CLASSIFICATIONS (First 10 Pages)")
print("="*80)

for i, sample in enumerate(separation_stats['sample_classifications'][:10], 1):
    print(f"\n{i}. {sample['pdf']} - Page {sample['page']}")
    print(f"   Language: {sample['language'].upper()}")
    print(f"   Confidence: {sample['confidence']:.1%}")
    print(f"   Reason: {sample['reason']}")
    print(f"   Text: {sample['text_preview']}...")
    print("-"*80)


SAMPLE CLASSIFICATIONS (First 10 Pages)

1. අංගුත්තර_නිකාය_1 - Page 1
   Language: SINHALA
   Confidence: 79.1%
   Reason: Strong Sinhala (79.1%)
   Text: නමෝ තස්ස භගවතෝ සබ්බධම්මේසු අප්පටිහතඤාණචාරස්ස දසබලධරස්ස චතුවේසාරජ්ජවිසාරදස්ස සබ්බසත්තුත්තමස්ස ධම්මිස්සරස්ස ධම්මරාජස්ස ධම්මස්සාමිස්ස තථාගතස්ස සබ්බඤ්ඤුනො...
--------------------------------------------------------------------------------

2. අංගුත්තර_නිකාය_1 - Page 2
   Language: SINHALA
   Confidence: 91.3%
   Reason: Strong Sinhala (91.3%)
   Text: සංඥාපනය අංගුත්තර නිකාය අප බුදුන් බරණැස්නුවර ඍෂීන් මුළු දෙන මුවළැවහි දී ධර්ම චක්‍ර'ය පැවැත් වූ තැන් පටන් කොට සුභද්‍ර පරිව්‍රාජක දමනය තෙක් පන්සාලිස් අවු...
--------------------------------------------------------------------------------

3. අංගුත්තර_නිකාය_1 - Page 3
   Language: SINHALA
   Confidence: 86.7%
   Reason: Strong Sinhala (86.7%)
   Text: හේතු උපමාදීන් ප්‍රතිමණ්ඩිත නානාවිධ දේශනානය සහස්‍රයන් විසින් විචිත්‍ර වන හෙයින් 'විචිත්‍ර ප්‍රතිභාජනන ය තෙල සඟියෙහි අනන්‍ය සාධාරණ ගුණයෙකි යි කියූ

## 12. Save Final Corpora

In [16]:
print("\nSaving final corpora...")

# Save Sinhala corpus
with open(SINHALA_CORPUS_FILE, 'w', encoding='utf-8') as f:
    f.write("\n\n".join(sinhala_texts))
print(f"✓ Sinhala corpus: {SINHALA_CORPUS_FILE}")
print(f"   Pages: {len(sinhala_texts):,}")
print(f"   Size: {SINHALA_CORPUS_FILE.stat().st_size / (1024*1024):.2f} MB")

# Save Pali corpus
with open(PALI_CORPUS_FILE, 'w', encoding='utf-8') as f:
    f.write("\n\n".join(pali_texts))
print(f"\n✓ Pali corpus: {PALI_CORPUS_FILE}")
print(f"   Pages: {len(pali_texts):,}")
print(f"   Size: {PALI_CORPUS_FILE.stat().st_size / (1024*1024):.2f} MB")

# Save mixed corpus
if mixed_texts:
    with open(MIXED_CORPUS_FILE, 'w', encoding='utf-8') as f:
        f.write("\n\n".join(mixed_texts))
    print(f"\n✓ Mixed corpus: {MIXED_CORPUS_FILE}")
    print(f"   Pages: {len(mixed_texts):,}")
    print(f"   Size: {MIXED_CORPUS_FILE.stat().st_size / (1024*1024):.2f} MB")


Saving final corpora...
✓ Sinhala corpus: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/preprocessed/sinhala_text_corpus.txt
   Pages: 10,259
   Size: 41.26 MB

✓ Pali corpus: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/preprocessed/pali_text_corpus.txt
   Pages: 252
   Size: 1.01 MB


## 13. Final Statistics

In [17]:
def calculate_word_stats(texts: List[str]) -> Dict:
    total_words = sum(len(text.split()) for text in texts)
    total_chars = sum(len(text) for text in texts)
    return {
        'total_words': total_words,
        'total_characters': total_chars,
        'num_pages': len(texts)
    }

sinhala_stats = calculate_word_stats(sinhala_texts)
pali_stats = calculate_word_stats(pali_texts)
mixed_stats = calculate_word_stats(mixed_texts)

print("\n" + "="*80)
print("FINAL CORPUS STATISTICS")
print("="*80)

print(f"\n📚 Sinhala Corpus:")
print(f"   Pages: {sinhala_stats['num_pages']:,}")
print(f"   Words: {sinhala_stats['total_words']:,}")
print(f"   Characters: {sinhala_stats['total_characters']:,}")

print(f"\n📚 Pali Corpus:")
print(f"   Pages: {pali_stats['num_pages']:,}")
print(f"   Words: {pali_stats['total_words']:,}")
print(f"   Characters: {pali_stats['total_characters']:,}")

if mixed_texts:
    print(f"\n📚 Mixed Corpus:")
    print(f"   Pages: {mixed_stats['num_pages']:,}")
    print(f"   Words: {mixed_stats['total_words']:,}")
    print(f"   Characters: {mixed_stats['total_characters']:,}")

total_words = sinhala_stats['total_words'] + pali_stats['total_words'] + mixed_stats['total_words']
print(f"\n📊 Total Words: {total_words:,}")
if total_words > 0:
    print(f"   Sinhala: {sinhala_stats['total_words']/total_words*100:.1f}%")
    print(f"   Pali: {pali_stats['total_words']/total_words*100:.1f}%")
    if mixed_texts:
        print(f"   Mixed: {mixed_stats['total_words']/total_words*100:.1f}%")


FINAL CORPUS STATISTICS

📚 Sinhala Corpus:
   Pages: 10,259
   Words: 2,599,498
   Characters: 16,679,794

📚 Pali Corpus:
   Pages: 252
   Words: 55,659
   Characters: 401,943

📊 Total Words: 2,655,157
   Sinhala: 97.9%
   Pali: 2.1%


## 14. Comprehensive Report

In [18]:
summary_report = f"""
BUDDHIST CORPUS PREPROCESSING SUMMARY (v4)
{'='*80}

RAW CORPUS (Before Preprocessing):
  - Total PDFs: {raw_stats['num_pdfs']}
  - Total Pages: {raw_stats['num_pages']:,}
  - Total Words: {raw_stats['total_words']:,}
  - Total Characters: {raw_stats['total_characters']:,}

FINAL CORPORA (After Language Separation):

  Sinhala Corpus:
    - Pages: {sinhala_stats['num_pages']:,}
    - Words: {sinhala_stats['total_words']:,} ({sinhala_stats['total_words']/total_words*100:.1f}%)
    - Characters: {sinhala_stats['total_characters']:,}
    - File: sinhala_text_corpus.txt

  Pali Corpus:
    - Pages: {pali_stats['num_pages']:,}
    - Words: {pali_stats['total_words']:,} ({pali_stats['total_words']/total_words*100:.1f}%)
    - Characters: {pali_stats['total_characters']:,}
    - File: pali_text_corpus.txt

CLASSIFICATION METHOD:
  - Pattern-based grammatical analysis
  - Pali indicators: -තෝ, -ස්ස, -නො, -ති, -න්ති endings
  - Sinhala indicators: වූ, හි, ට, ගේ particles
  - Confidence threshold: {CLASSIFICATION_THRESHOLD}

NOTES:
  - Ready for SinLlama perplexity evaluation
  - Buddhist doctrinal text may show higher perplexity
  - Pali transliterations separated for specialized analysis

{'='*80}
"""

print(summary_report)

# Save report
with open(ANALYSIS_DIR / "preprocessing_summary_v4.txt", 'w', encoding='utf-8') as f:
    f.write(summary_report)

print(f"\n✓ Summary saved to: {ANALYSIS_DIR / 'preprocessing_summary_v4.txt'}")


BUDDHIST CORPUS PREPROCESSING SUMMARY (v4)

RAW CORPUS (Before Preprocessing):
  - Total PDFs: 34
  - Total Pages: 10,511
  - Total Words: 2,753,232
  - Total Characters: 17,819,309

FINAL CORPORA (After Language Separation):
  
  Sinhala Corpus:
    - Pages: 10,259
    - Words: 2,599,498 (97.9%)
    - Characters: 16,679,794
    - File: sinhala_text_corpus.txt
  
  Pali Corpus:
    - Pages: 252
    - Words: 55,659 (2.1%)
    - Characters: 401,943
    - File: pali_text_corpus.txt

CLASSIFICATION METHOD:
  - Pattern-based grammatical analysis
  - Pali indicators: -තෝ, -ස්ස, -නො, -ති, -න්ති endings
  - Sinhala indicators: වූ, හි, ට, ගේ particles
  - Confidence threshold: 0.6

NOTES:
  - Ready for SinLlama perplexity evaluation
  - Buddhist doctrinal text may show higher perplexity
  - Pali transliterations separated for specialized analysis



✓ Summary saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/preprocessed/analysis/preprocessing_summ

## Summary

**v4 Improvements:**
- ✅ Robust pattern-based Sinhala/Pali classifier
- ✅ No dependency on external models (fastText)
- ✅ Extensive grammatical pattern matching
- ✅ Confidence scoring for all classifications
- ✅ Sample output for verification
- ✅ Adjustable classification threshold

**Files Created:**
- `sinhala_text_corpus.txt` - For SinLlama perplexity evaluation
- `pali_text_corpus.txt` - Pali transliterations
- `mixed_text_corpus.txt` - Uncertain classifications
- Analysis and statistics in `analysis/` folder